# 05 — Combined Cross-Tissue Report

Aggregate per-tissue subcluster reports (produced by `02_subcluster_cards.ipynb`) into a
single interactive HTML report and a combined PowerPoint deck.

**Workflow:**
1. For each tissue, locate the `{TISSUE}_subcluster_report/` directory from Notebook 02
2. Copy all `img/` and `de/` assets into a unified `OUTPUT_DIR/img/{tissue}/` layout
3. Reconstruct card metadata from saved CSV and image file lists
4. Render `combined_report.html.j2` → `OUTPUT_DIR/index.html`
5. Build a combined PPTX with tissue-title slides between subcluster slides

**Outputs (`OUTPUT_DIR/`):**
- `index.html` — interactive combined report (keep `img/` and `de/` alongside it)
- `img/{tissue}/{safe}/` — per-tissue/per-subcluster PNG files
- `de/{tissue}_{safe}.csv` — DE tables (downloadable)
- `combined_report.pptx` — combined PowerPoint deck

## 1. Imports

In [ ]:
import io
import json
import re
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
from jinja2 import Environment, FileSystemLoader
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm, Inches, Pt

print('Imports OK')

## 2. Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Map tissue name → path to the {TISSUE}_subcluster_report/ directory
# produced by Notebook 02 for that tissue.
TISSUE_REPORT_DIRS: dict[str, Path] = {
    "lung":  Path("results/disease_subclusters/lung/lung_subcluster_report"),
    "brain": Path("results/disease_subclusters/brain/brain_subcluster_report"),
    # add more tissues here …
}

# Where to write the combined report (img/ and de/ will be created inside)
OUTPUT_DIR: Path = Path("results/disease_subclusters/combined_report")

# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'img').mkdir(exist_ok=True)
(OUTPUT_DIR / 'de').mkdir(exist_ok=True)

print('Configuration loaded.')
for tissue, rdir in TISSUE_REPORT_DIRS.items():
    exists = '✓' if rdir.exists() else '✗ NOT FOUND'
    print(f'  {tissue}: {rdir}  {exists}')

## 3. Copy Assets & Reconstruct Card Metadata

In [ ]:

# ---------------------------------------------------------------------------
# Safe-label helpers
# ---------------------------------------------------------------------------

_SAFE_LABEL_CACHE: dict[str, str] = {}


def _abbrev_words(text: str, chars_per_word: int = 3) -> str:
    """Return first ``chars_per_word`` chars of each whitespace-separated word."""
    return "_".join(w[:chars_per_word] for w in text.split() if w)


def _safe_label_raw(label: str, max_len: int = 35) -> str:
    if "|" in label:
        parts = [p.strip() for p in label.split("|")]
        ct_abbrev = re.sub(r"[^\w\-]", "_", _abbrev_words(parts[0]))
        ct_abbrev = re.sub(r"_+", "_", ct_abbrev).strip("_")
        rest_parts = [re.sub(r"[^\w\-]", "_", p) for p in parts[1:]]
        suffix = "_".join(rest_parts)
        safe = f"{ct_abbrev}_{suffix}"
        if len(safe) > max_len and len(rest_parts) >= 2:
            sub_id = rest_parts[-1]
            disease = "_".join(rest_parts[:-1])
            budget = max_len - len(ct_abbrev) - 2 - len(sub_id)
            disease = disease[:max(0, budget)].rstrip("_")
            safe = f"{ct_abbrev}_{disease}_{sub_id}"
    else:
        safe = re.sub(r"[^\w\-]", "_", label)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe[:max_len].rstrip("_")


def _safe_label(label: str) -> str:
    if label in _SAFE_LABEL_CACHE:
        return _SAFE_LABEL_CACHE[label]
    return _safe_label_raw(label)


def _build_safe_label_map(labels: list[str]) -> None:
    _SAFE_LABEL_CACHE.clear()
    used: set[str] = set()
    for label in labels:
        base = _safe_label_raw(label)
        if base not in used:
            _SAFE_LABEL_CACHE[label] = base
            used.add(base)
        else:
            i = 2
            stem = base[:32].rstrip("_")
            candidate = f"{stem}_{i}"
            while candidate in used:
                i += 1
                candidate = f"{stem}_{i}"
            _SAFE_LABEL_CACHE[label] = candidate
            used.add(candidate)
    print(f"Safe-label map built: {len(labels)} subclusters.")


# ---------------------------------------------------------------------------
# Stats helpers (mirrors NB02 _parse_stats / _parse_disease_breakdown)
# ---------------------------------------------------------------------------

def _fmt_float(v, decimals: int = 3) -> str:
    try:
        return f"{float(v):.{decimals}g}"
    except (ValueError, TypeError):
        return str(v)


def _parse_stats_from_row(r, info_df: pd.DataFrame) -> dict:
    """Extract the full stats dict from a subcluster_info row."""
    n_total   = int(r["n_cells"])         if pd.notna(r.get("n_cells"))         else "—"
    n_disease = int(r["n_disease_cells"]) if pd.notna(r.get("n_disease_cells")) else "—"
    n_healthy = (
        n_total - n_disease
        if isinstance(n_total, int) and isinstance(n_disease, int)
        else "—"
    )

    if "disease_composition" in info_df.columns and pd.notna(r.get("disease_composition")):
        disease_composition = str(r["disease_composition"])
    else:
        diseases_str = (
            r.get("diseases_contributing", "")
            if "diseases_contributing" in info_df.columns
            else ""
        )
        if diseases_str and str(diseases_str).strip():
            disease_composition = str(diseases_str)
        else:
            disease_composition = str(r.get("disease", "—"))

    return {
        "n_total":             str(n_total),
        "n_disease":           str(n_disease),
        "n_healthy":           str(n_healthy),
        "fold_enrichment":     _fmt_float(r.get("fold_enrichment", "—")),
        "fdr_pval":            _fmt_float(r.get("pvalue_adj", "—"), decimals=2),
        "cell_type":           str(r.get("subset", "—")),
        "disease_composition": disease_composition,
        "de_reference":        "—",  # filled in separately
    }


def _parse_breakdown_from_row(r, info_df: pd.DataFrame) -> list[dict]:
    """Reconstruct per-disease breakdown from a subcluster_info row."""
    diseases_str = (
        r.get("diseases_contributing", "")
        if "diseases_contributing" in info_df.columns
        else ""
    )
    if diseases_str and str(diseases_str).strip():
        diseases = [d.strip() for d in str(diseases_str).split(",") if d.strip()]
        return [{"disease": d, "n_cells": "—", "pct": "—", "fold": "—", "pval": "—", "fdr": "—"}
                for d in sorted(diseases)]
    disease_label = str(r.get("disease", ""))
    if disease_label and disease_label not in ("combined", "nan", "—"):
        return [{"disease": disease_label,
                 "n_cells": str(int(r["n_disease_cells"])) if pd.notna(r.get("n_disease_cells")) else "—",
                 "pct": "—",
                 "fold": _fmt_float(r.get("fold_enrichment", "—")),
                 "pval": _fmt_float(r.get("pvalue", "—")),
                 "fdr":  _fmt_float(r.get("pvalue_adj", "—"), decimals=2)}]
    return []


def _get_de_reference_from_dir(label: str, de_dir: Path) -> str:
    """Extract DE reference group from DE CSV filenames/contents in de_dir."""
    if not de_dir.exists():
        return "—"

    safe = _safe_label(label)

    # 3-level search matching NB02 _get_de_reference
    matches = [p for p in sorted(de_dir.glob(f"de_{safe}*.csv"))
               if "all_comparisons" not in p.name]
    if not matches:
        matches = [p for p in sorted(de_dir.glob("de_*.csv"))
                   if safe in p.stem and "all_comparisons" not in p.name]
    if not matches:
        _old_safe = re.sub(r'[^\w\-]', '_', label.replace('|', '_'))
        _old_safe = re.sub(r'_+', '_', _old_safe).strip('_')
        matches = [p for p in sorted(de_dir.glob(f'de_{_old_safe}*.csv'))
                   if 'all_comparisons' not in p.name]
    if not matches:
        return "—"

    refs: list[str] = []
    for p in matches:
        ref_str = None
        try:
            df_peek = pd.read_csv(p, nrows=1)
            if "comparison" in df_peek.columns:
                comp = str(df_peek["comparison"].iloc[0])
                if "_vs_" in comp:
                    ref_str = comp.split("_vs_", 1)[1]
        except Exception:
            pass
        if ref_str is None:
            stem = p.stem.removeprefix("de_")
            ref_str = stem.split("_vs_", 1)[1] if "_vs_" in stem else None
        refs.append(ref_str or "rest")

    seen: set[str] = set()
    unique_refs = [r for r in refs if not (r in seen or seen.add(r))]
    return ", ".join(unique_refs) if unique_refs else "—"


# ---------------------------------------------------------------------------
# Asset copy + card reconstruction
# ---------------------------------------------------------------------------

def _copy_subcluster_assets(
    tissue: str,
    report_dir: Path,
    out_dir: Path,
) -> list[dict]:
    """
    Copy img/{safe}/ and de/ from a per-tissue report dir into
    out_dir/img/{tissue}/{safe}/ and out_dir/de/{tissue}_{safe}.csv.
    Returns a list of card metadata dicts (one per subcluster).
    """
    cards: list[dict] = []

    img_src = report_dir / 'img'
    de_src  = report_dir / 'de'

    if not img_src.exists():
        print(f'  [warn] No img/ dir in {report_dir}')
        return cards

    for safe_dir in sorted(img_src.iterdir()):
        if not safe_dir.is_dir():
            continue
        safe = safe_dir.name

        dst_img = out_dir / 'img' / tissue / safe
        dst_img.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(safe_dir.glob('*.png')):
            shutil.copy2(src_file, dst_img / src_file.name)

        def _img(name, _safe=safe, _tissue=tissue):
            p = dst_img / name
            return f'img/{_tissue}/{_safe}/{name}' if p.exists() else None

        de_csv_path = None
        de_json = None
        if de_src.exists():
            src_csv = de_src / f'{safe}.csv'
            if src_csv.exists():
                dst_csv = out_dir / 'de' / f'{tissue}_{safe}.csv'
                shutil.copy2(src_csv, dst_csv)
                de_csv_path = f'de/{tissue}_{safe}.csv'
                try:
                    df = pd.read_csv(dst_csv)
                    de_json = df.to_json(orient='split')
                except Exception:
                    de_json = None

        card = {
            'label':             safe.replace('_', ' '),
            'anchor':            f'{tissue}_{safe}',
            'disease':           safe,
            'stats':             {},
            'disease_breakdown': [],
            'umap_celltype_path':  _img('umap_celltype.png'),
            'umap_disease_path':   _img('umap_disease.png'),
            'umap_highlight_path': _img('umap_highlight.png'),
            'volcano_path':        _img('volcano.png'),
            'dotplot_path':        _img('enrichment.png'),
            'de_csv_path':         de_csv_path,
            'de_json':             de_json,
            '_safe':               safe,   # internal; used for label lookup
        }
        cards.append(card)

    # ── Enrich from subcluster_info CSV ──────────────────────────────────
    info_csv = report_dir.parent / 'subcluster_info.csv'
    if not info_csv.exists():
        matches = list(report_dir.parent.glob('*_subcluster_info.csv'))
        if matches:
            info_csv = matches[0]

    if info_csv.exists():
        try:
            info_df = pd.read_csv(info_csv)
            if 'label' in info_df.columns:
                # Build safe → original-label map for this tissue
                _build_safe_label_map([str(lbl) for lbl in info_df['label']])
                safe_to_label = {
                    _safe_label(str(row['label'])): str(row['label'])
                    for _, row in info_df.iterrows()
                }
                for card in cards:
                    label = safe_to_label.get(card['_safe'], card['label'])
                    card['label'] = label
                    card['disease'] = label.split('|')[1] if '|' in label else label

                    row = info_df[info_df['label'] == label]
                    if not row.empty:
                        r = row.iloc[0]
                        card['stats'] = _parse_stats_from_row(r, info_df)
                        card['disease_breakdown'] = _parse_breakdown_from_row(r, info_df)
                        card['stats']['de_reference'] = _get_de_reference_from_dir(
                            label, de_src
                        )
                        # Rebuild disease_composition from breakdown if generic
                        _bd = card['disease_breakdown']
                        _comp = card['stats'].get('disease_composition', '')
                        if _bd and (not _comp or _comp.lower() in ('combined', '—', 'unknown', 'nan')):
                            card['stats']['disease_composition'] = ', '.join(
                                f"{r2['disease']}: {r2['n_cells']} ({r2['pct']})"
                                for r2 in _bd
                            )
        except Exception as e:
            print(f'  [warn] Could not read subcluster_info from {info_csv}: {e}')

    # Clean up internal key
    for card in cards:
        card.pop('_safe', None)

    return cards


# ── Main loop ──────────────────────────────────────────────────────────────
tissues: dict[str, list[dict]] = {}

for tissue, report_dir in TISSUE_REPORT_DIRS.items():
    if not report_dir.exists():
        print(f'⚠  {tissue}: report directory not found — skipping ({report_dir})')
        continue
    print(f'\n── {tissue}')
    cards = _copy_subcluster_assets(tissue, report_dir, OUTPUT_DIR)
    tissues[tissue] = cards
    print(f'  {len(cards)} subclusters copied')

total_subclusters = sum(len(v) for v in tissues.values())
print(f'\n✓ {len(tissues)} tissue(s), {total_subclusters} subclusters total.')
print(f'  Output dir: {OUTPUT_DIR.resolve()}')


## 4. Render Combined HTML Report

In [ ]:
if '__file__' in dir():
    TEMPLATE_DIR = Path(__file__).parent / 'templates'
elif (Path.cwd() / 'templates').is_dir():
    TEMPLATE_DIR = Path('templates')
else:
    TEMPLATE_DIR = Path('notebooks/disease_subclusters/templates')

env      = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)), autoescape=False)
template = env.get_template('combined_report.html.j2')

html_out = template.render(
    tissues=tissues,
    total_subclusters=total_subclusters,
    generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'),
)

html_path = OUTPUT_DIR / 'index.html'
html_path.write_text(html_out, encoding='utf-8')
size_kb = html_path.stat().st_size / 1024
print(f'✓ Combined HTML written: {html_path}  ({size_kb:.0f} KB)')
print(f'  Open: {html_path.resolve()}')

## 5. Build Combined PowerPoint Deck

In [ ]:
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

_DARK  = RGBColor(0x1a, 0x1a, 0x2e)
_BLUE  = RGBColor(0x4a, 0x90, 0xd9)
_WHITE = RGBColor(0xff, 0xff, 0xff)
_LGREY = RGBColor(0xf0, 0xf2, 0xf5)
_TEXT  = RGBColor(0x1a, 0x1a, 0x2e)

# Left panel
_LEFT_L = Cm(0.4)
_LEFT_W = Cm(12.7)
# Right panel
_RIGHT_L = Cm(13.5)


def _add_text_box(slide, left, top, width, height, text, *,
                  bold=False, fontsize=10, color=_TEXT,
                  align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = wrap
    p = tf.paragraphs[0]
    p.text = text
    p.alignment = align
    run = p.runs[0] if p.runs else p.add_run()
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color
    return txBox


def _set_cell_text(cell, text: str, *, bold: bool = False, fontsize: int = 6,
                   bg_color: RGBColor | None = None,
                   color: RGBColor = _TEXT):
    """Set text and font on a pptx table cell."""
    if bg_color is not None:
        cell.fill.solid()
        cell.fill.fore_color.rgb = bg_color
    tf = cell.text_frame
    tf.word_wrap = False
    p = tf.paragraphs[0]
    p.clear()
    run = p.add_run()
    run.text = text
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color


def _add_table_section_title(slide, left, top, width, text):
    """Add a small uppercase section label above a table; returns new top."""
    _add_text_box(slide, left, top, width, Cm(0.42),
                  text, bold=True, fontsize=7, color=_BLUE)
    return top + Cm(0.42)


def _add_stats_table(slide, stats, breakdown, top=Cm(2.0),
                     left=Cm(0.4), width=Cm(12.7)):
    """Render subcluster stats as a two-column pptx table. Returns bottom y."""
    rows_data = [
        ("Total cells",         str(stats.get("n_total",             "—"))),
        ("Disease cells",       str(stats.get("n_disease",           "—"))),
        ("Healthy cells",       str(stats.get("n_healthy",           "—"))),
        ("Fold enrichment",     str(stats.get("fold_enrichment",     "—"))),
        ("FDR p-value",         str(stats.get("fdr_pval",            "—"))),
        ("Cell type",           str(stats.get("cell_type",           "—"))),
        ("Disease composition", str(stats.get("disease_composition", "—"))),
        ("DE reference",        str(stats.get("de_reference",        "—"))),
    ]
    if breakdown:
        for r in breakdown:
            rows_data.append((
                f"  ↳ {r['disease']}",
                f"n={r['n_cells']} ({r['pct']})"
                + (f"  FDR={r['fdr']}" if r['fdr'] != "—" else ""),
            ))

    tbl_top = _add_table_section_title(slide, left, top, width, "SUMMARY STATISTICS")
    row_h = Cm(0.42)
    tbl = slide.shapes.add_table(
        len(rows_data), 2, left, tbl_top, width, row_h * len(rows_data)
    ).table
    tbl.columns[0].width = Cm(4.5)
    tbl.columns[1].width = width - Cm(4.5)
    for ri, (lbl, val) in enumerate(rows_data):
        _set_cell_text(tbl.cell(ri, 0), lbl, bold=True,  fontsize=6, bg_color=_LGREY)
        _set_cell_text(tbl.cell(ri, 1), val, bold=False, fontsize=6)
    return tbl_top + row_h * len(rows_data)


def _add_de_table(slide, de_json, top, left=Cm(0.4), width=Cm(12.7),
                  pval_cutoff=0.05, lfc_cutoff=0.5, top_n=20):
    """Render top-N significant upregulated DE genes as a pptx table with title."""
    if not de_json:
        return
    try:
        de_df = pd.read_json(io.StringIO(de_json), orient='split')
    except Exception:
        return
    if de_df.empty:
        return

    # Detect column names flexibly
    pval_col = next((c for c in ["padj", "pvals_adj", "pvalue", "pvals"] if c in de_df.columns), None)
    lfc_col  = next((c for c in ["log2FoldChange", "logfoldchanges"] if c in de_df.columns), None)
    gene_col = next((c for c in ["gene", "gene_id"] if c in de_df.columns), None)

    up_df = de_df.copy()
    if pval_col:
        up_df = up_df[pd.to_numeric(up_df[pval_col], errors='coerce') < pval_cutoff]
    if lfc_col:
        up_df = up_df[pd.to_numeric(up_df[lfc_col], errors='coerce') > lfc_cutoff]
    if pval_col and not up_df.empty:
        up_df = up_df.sort_values(pval_col)
    if up_df.empty:
        return

    tbl_top = _add_table_section_title(slide, left, top, width, "TOP UPREGULATED GENES")

    preferred = [gene_col, lfc_col, pval_col, "baseMean", "gene_id"]
    show_cols = [c for c in preferred if c and c in up_df.columns][:5]
    if not show_cols:
        show_cols = list(up_df.columns[:5])

    sub = up_df[show_cols].head(top_n).fillna("").copy()
    _pval_cols = {"padj", "pvalue", "pvals_adj", "pvals"}
    for c in sub.select_dtypes(include="float").columns:
        if c in _pval_cols:
            sub[c] = sub[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) and v != "" else "")
        else:
            sub[c] = sub[c].round(4).astype(str)

    n_rows = len(sub) + 1
    n_cols = len(show_cols)
    row_h  = Cm(0.42)
    tbl = slide.shapes.add_table(
        n_rows, n_cols, left, tbl_top, width, row_h * n_rows
    ).table
    col_w = width // n_cols
    for ci in range(n_cols):
        tbl.columns[ci].width = col_w
    for ci, col in enumerate(show_cols):
        _set_cell_text(tbl.cell(0, ci), col, bold=True, fontsize=6, bg_color=_LGREY)
    for ri, (_, row) in enumerate(sub.iterrows(), start=1):
        for ci, col in enumerate(show_cols):
            _set_cell_text(tbl.cell(ri, ci), str(row[col]), fontsize=6)


def _add_tissue_title_slide(prs, tissue: str, n_subclusters: int):
    """Insert a tissue-level divider slide."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, SLIDE_W, SLIDE_H)
    bg.fill.solid()
    bg.fill.fore_color.rgb = _DARK
    bg.line.fill.background()
    _add_text_box(
        slide, Cm(4), Cm(2.5), Cm(25), Cm(3),
        tissue.upper(), bold=True, fontsize=36, color=_WHITE, align=PP_ALIGN.CENTER,
    )
    _add_text_box(
        slide, Cm(4), Cm(6), Cm(25), Cm(1.5),
        f'{n_subclusters} disease-enriched subcluster(s)',
        bold=False, fontsize=16, color=_BLUE, align=PP_ALIGN.CENTER,
    )


def _add_subcluster_slide(prs, tissue: str, card: dict):
    """Add a single subcluster slide matching the NB02 per-tissue layout."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    # Title bar
    bar = slide.shapes.add_shape(1, 0, 0, SLIDE_W, Cm(1.8))
    bar.fill.solid()
    bar.fill.fore_color.rgb = _DARK
    bar.line.fill.background()
    _add_text_box(slide, Cm(0.4), Cm(0.1), Cm(20), Cm(1.6),
                  card['label'], bold=True, fontsize=14, color=_WHITE)
    _add_text_box(slide, Cm(21), Cm(0.1), Cm(12), Cm(1.6),
                  f'Tissue: {tissue}', bold=False, fontsize=11, color=_BLUE,
                  align=PP_ALIGN.RIGHT)

    # Left panel: stats + DE table
    stats_bottom = _add_stats_table(
        slide,
        card.get('stats', {}),
        card.get('disease_breakdown', []),
    )
    _add_de_table(slide, card.get('de_json'), top=stats_bottom + Cm(1.2))

    # Right panel — row 1: 3× UMAP
    umap_top = Cm(2.0)
    umap_h   = Cm(7.2)
    umap_w   = Cm(6.5)
    gap      = Cm(0.18)

    for i, key in enumerate(['umap_celltype_path', 'umap_disease_path', 'umap_highlight_path']):
        img_rel = card.get(key)
        if img_rel:
            img_path = OUTPUT_DIR / img_rel
            if img_path.exists():
                try:
                    slide.shapes.add_picture(
                        str(img_path),
                        _RIGHT_L + i * (umap_w + gap), umap_top,
                        width=umap_w,
                    )
                except Exception:
                    pass

    # Right panel — row 2: volcano + dotplot
    plot_top = umap_top + umap_h + Cm(0.25)
    plot_w   = Cm(10.0)

    for i, key in enumerate(['volcano_path', 'dotplot_path']):
        img_rel = card.get(key)
        if img_rel:
            img_path = OUTPUT_DIR / img_rel
            if img_path.exists():
                try:
                    slide.shapes.add_picture(
                        str(img_path),
                        _RIGHT_L + i * (plot_w + gap), plot_top,
                        width=plot_w,
                    )
                except Exception:
                    pass


# ── Build presentation ───────────────────────────────────────────────────────
prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H

for tissue, cards in tissues.items():
    _add_tissue_title_slide(prs, tissue, len(cards))
    for card in cards:
        _add_subcluster_slide(prs, tissue, card)

pptx_path = OUTPUT_DIR / 'combined_report.pptx'
prs.save(str(pptx_path))
print(f'✓ Combined PPTX written: {pptx_path}  ({pptx_path.stat().st_size / 1024:.0f} KB)')


## Summary

| Output | Path |
|--------|------|
| Combined HTML report | `{OUTPUT_DIR}/index.html` |
| Per-tissue images | `{OUTPUT_DIR}/img/{tissue}/{safe}/` |
| DE tables (CSV) | `{OUTPUT_DIR}/de/{tissue}_{safe}.csv` |
| Combined PowerPoint | `{OUTPUT_DIR}/combined_report.pptx` |

**Viewing the report:** open `index.html` in a browser — the `img/` and `de/` folders must stay alongside it.  
**Sharing:** zip the entire `{OUTPUT_DIR}/` directory; all paths are relative.